# Exit Stage

An `ExitStage` is the simplest destination in JuPedSim: agents head straight for it and are removed from the simulation the moment they step inside its polygon.
It sits at the end of every journey and is created with `simulation.add_exit_stage(polygon_vertices)`.
This notebook shows the minimal case: a single room, one exit, twenty agents.

See {doc}`Object model & lifecycle </concepts/object_model>` for how stages fit into the simulation lifecycle.

In [ ]:
import pathlib

import jupedsim as jps
import shapely

# 10 m x 10 m room.
geometry = shapely.Polygon([(0, 0), (10, 0), (10, 10), (0, 10)])

trajectory_file = pathlib.Path("exits.sqlite")
simulation = jps.Simulation(
    model=jps.CollisionFreeSpeedModel(),
    geometry=geometry,
    trajectory_writer=jps.SqliteTrajectoryWriter(output_file=trajectory_file),
)

# One exit on the right wall — agents are removed when they reach it.
exit_id = simulation.add_exit_stage([(9, 4), (10, 4), (10, 6), (9, 6)])
journey_id = simulation.add_journey(jps.JourneyDescription([exit_id]))

# Place 20 agents on the left side of the room.
n_agents = 20
positions = jps.distributions.distribute_by_number(
    polygon=shapely.Polygon([(0.5, 0.5), (3, 0.5), (3, 9.5), (0.5, 9.5)]),
    number_of_agents=n_agents,
    distance_to_agents=0.4,
    distance_to_polygon=0.2,
    seed=1,
)
for position in positions:
    simulation.add_agent(
        jps.CollisionFreeSpeedModelAgentParameters(
            journey_id=journey_id,
            stage_id=exit_id,
            position=position,
            radius=0.12,
        )
    )

# Iterate until the room is empty.
while simulation.agent_count() > 0 and simulation.iteration_count() < 10_000:
    simulation.iterate()

print(
    f"Evacuated {n_agents} agents in {simulation.iteration_count()} iterations "
    f"({simulation.elapsed_time():.1f} s). Trajectories: {trajectory_file}"
)

## Watch it

In [ ]:
from jupedsim.internal.notebook_utils import animate, read_sqlite_file

trajectory_data, walkable_area = read_sqlite_file(trajectory_file)
animate(trajectory_data, walkable_area)

## Try one change

Move or widen the exit polygon — change `[(9, 4), (10, 4), (10, 6), (9, 6)]` to `[(8, 3), (10, 3), (10, 7), (8, 7)]` — and re-run. Notice how a wider opening reduces evacuation time.

## See also

- {doc}`Waypoint Stage </notebooks/fundamentals/04_waypoints>` — an intermediate target before the exit.
- {doc}`Journeys and Transitions </notebooks/fundamentals/07_journeys_transitions>` — chain stages together.
- {doc}`Object model & lifecycle </concepts/object_model>`.
- [jupedsim-scenarios](https://scenarios.jupedsim.org) — parameter sweeps and Monte-Carlo runs.